In [1]:
# Create chronological train, validation, test, and expanding-window walk-forward IDs.
# Walk-forward preprocessing must be refit on each fold training set only.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, file_hash
from project_config import ID_COLUMNS
from pathlib import Path
import ast
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/modeling_dataset_unprocessed.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
START_YEAR = 1998
INITIAL_TRAIN_END_YEAR = 2015
INITIAL_TEST_START_YEAR = 2020
END_YEAR = 2026
MIN_BANKRUPTCIES = {"train": 50, "validation": 20, "test": 20}
MIN_CONTROLS = {"train": 20, "validation": 20, "test": 20}
MIN_NOMINAL_TRAIN_YEARS = 5
SPLIT_NAMES = ["train", "validation", "test"]
KEYS = ["company_id", "prediction_date"]
input_hashes = {}


def event_config(variable):
    path = ROOT / "notebooks/05_create_bankruptcy_event_dates.ipynb"
    input_hashes[path] = file_hash(path)
    notebook = json.loads(path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, ast.Assign) and any(
                isinstance(target, ast.Name) and target.id == variable
                for target in node.targets
            ):
                value = node.value
                if (
                    isinstance(value, ast.Call)
                    and isinstance(value.func, ast.Attribute)
                    and isinstance(value.func.value, ast.Name)
                    and value.func.value.id == "pd"
                    and value.func.attr == "Timestamp"
                    and value.args
                ):
                    value = value.args[0]
                return ast.literal_eval(value)
    raise ValueError(f"Missing event configuration: {variable}")


model_path = PROCESSED / "modeling_dataset_unprocessed.csv"
input_hashes[model_path] = file_hash(model_path)
source = pd.read_csv(
    model_path, dtype={"company_id": "string", "metadata_event_id": "string"}
)
required = KEYS + [
    "target",
    "prediction_year",
    "metadata_event_id",
    "metadata_outcome_end_date",
    "metadata_bloomberg_identifier",
    "metadata_matching_records_json",
]
missing = sorted(set(required) - set(source.columns))
if source.empty or missing:
    raise ValueError(f"Empty model data or missing columns: {missing}")
if source[KEYS + ["metadata_event_id", "target", "prediction_year"]].isna().any().any():
    raise ValueError("Missing event identifiers, dates, or labels")
if source.duplicated(KEYS).any() or not source.metadata_event_id.is_unique:
    raise ValueError("Require one row per unique company-event")
if not source.target.isin([0, 1]).all():
    raise ValueError("Target must be binary")
source_before = source.copy(deep=True)
events = source[required].copy()
events["prediction_date"] = pd.to_datetime(
    events.prediction_date, errors="raise"
).astype("datetime64[ns]")
events["metadata_outcome_end_date"] = pd.to_datetime(
    events.metadata_outcome_end_date, errors="raise"
).astype("datetime64[ns]")
if not events.prediction_year.eq(events.prediction_date.dt.year).all():
    raise ValueError("Prediction years disagree with dates")
if not events.prediction_date.eq(events.prediction_date.dt.normalize()).all():
    raise ValueError("Expected normalized daily prediction dates")
OUTCOME_MONTHS = 12
if not events.metadata_outcome_end_date.eq(
    events.prediction_date + pd.DateOffset(months=OUTCOME_MONTHS)
).all():
    raise ValueError(
        "Outcome dates must equal prediction plus the configured calendar horizon"
    )
VERIFIED_LABEL_END_DATE = pd.Timestamp(
    event_config("VERIFIED_LABEL_END_DATE")
).normalize()
events["outcome_complete"] = (
    events.metadata_outcome_end_date.notna()
    & events.metadata_outcome_end_date.le(VERIFIED_LABEL_END_DATE)
)

In [3]:
# Collapse company/security aliases to stable groups, without consulting labels/features.
def company_groups(frame):
    ids = sorted(frame.company_id.unique())
    parent = {company: company for company in ids}

    def find(company):
        while parent[company] != company:
            parent[company] = parent[parent[company]]
            company = parent[company]
        return company

    def unite(left, right):
        left, right = find(left), find(right)
        if left != right:
            parent[max(left, right)] = min(left, right)

    securities = (
        frame.metadata_bloomberg_identifier.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )
    aliases = pd.DataFrame({"company_id": frame.company_id, "security": securities})
    aliases = aliases.loc[
        aliases.security.notna() & aliases.security.ne("")
    ].drop_duplicates()
    for security, rows in aliases.groupby("security", sort=True):
        members = sorted(rows.company_id.unique())
        for member in members[1:]:
            unite(members[0], member)
    return frame.company_id.map(
        {company: "group:" + find(company) for company in ids}
    ).astype("string")


events["company_group_id"] = company_groups(events)


# Purge outcome overlap first, then assign each group to its latest eligible block.
def chronological_grouped_split(
    frame, start_year, train_end_year, test_start_year, end_year
):
    if not start_year <= train_end_year < test_start_year - 1 <= end_year - 1:
        raise ValueError("Invalid chronological year boundaries")
    audited = frame.copy()
    validation_start = pd.Timestamp(year=train_end_year + 1, month=1, day=1)
    test_start = pd.Timestamp(year=test_start_year, month=1, day=1)
    start = pd.Timestamp(year=start_year, month=1, day=1)
    end = pd.Timestamp(year=end_year + 1, month=1, day=1)
    in_window = audited.prediction_date.ge(start) & audited.prediction_date.lt(end)
    rank = np.select(
        [
            audited.prediction_date.lt(validation_start),
            audited.prediction_date.lt(test_start),
        ],
        [0, 1],
        default=2,
    )
    audited["nominal_split_rank"] = rank
    audited["nominal_split"] = pd.Series(rank, index=audited.index).map(
        dict(enumerate(SPLIT_NAMES))
    )
    audited["exclusion_reason"] = pd.Series(pd.NA, index=audited.index, dtype="string")
    audited.loc[~in_window, "exclusion_reason"] = "outside_configured_years"
    incomplete = audited.exclusion_reason.isna() & ~audited.outcome_complete
    audited.loc[incomplete, "exclusion_reason"] = "incomplete_outcome_window"
    crosses_boundary = (
        audited.nominal_split_rank.eq(0)
        & audited.metadata_outcome_end_date.ge(validation_start)
    ) | (
        audited.nominal_split_rank.eq(1)
        & audited.metadata_outcome_end_date.ge(test_start)
    )
    purge = audited.exclusion_reason.isna() & crosses_boundary
    audited.loc[purge, "exclusion_reason"] = "outcome_window_crosses_next_split"
    eligible = audited.exclusion_reason.isna()
    group_owner = (
        audited.loc[eligible].groupby("company_group_id").nominal_split_rank.max()
    )
    audited["owner_split_rank"] = audited.company_group_id.map(group_owner)
    group_exclusion = eligible & audited.nominal_split_rank.ne(audited.owner_split_rank)
    audited.loc[group_exclusion, "exclusion_reason"] = (
        "company_reserved_for_later_split"
    )
    audited["split"] = audited.nominal_split.where(audited.exclusion_reason.isna())
    return audited


def split_counts(audit):
    return pd.crosstab(
        audit.loc[audit.split.notna(), "split"],
        audit.loc[audit.split.notna(), "target"],
    ).reindex(index=SPLIT_NAMES, columns=[0, 1], fill_value=0)


def adequate(counts):
    return all(
        counts.loc[name, 1] >= MIN_BANKRUPTCIES[name]
        and counts.loc[name, 0] >= MIN_CONTROLS[name]
        for name in SPLIT_NAMES
    )


# Prefer the stated test period; consider an earlier test boundary only if needed.
# Expand validation by moving training end earlier, choosing the first adequate design.
initial_audit = chronological_grouped_split(
    events, START_YEAR, INITIAL_TRAIN_END_YEAR, INITIAL_TEST_START_YEAR, END_YEAR
)
initial_counts = split_counts(initial_audit)
if (
    initial_counts.loc["test", 1] >= MIN_BANKRUPTCIES["test"]
    and initial_counts.loc["test", 0] >= MIN_CONTROLS["test"]
):
    test_start_candidates = [INITIAL_TEST_START_YEAR]
else:
    test_start_candidates = range(
        INITIAL_TEST_START_YEAR, START_YEAR + MIN_NOMINAL_TRAIN_YEARS + 1, -1
    )
cutoff_trials = []
selected_audit = None
for test_start_year in test_start_candidates:
    largest_train_end = min(INITIAL_TRAIN_END_YEAR, test_start_year - 2)
    for train_end_year in range(
        largest_train_end, START_YEAR + MIN_NOMINAL_TRAIN_YEARS - 2, -1
    ):
        audit = chronological_grouped_split(
            events, START_YEAR, train_end_year, test_start_year, END_YEAR
        )
        counts = split_counts(audit)
        acceptable = adequate(counts)
        cutoff_trials.append(
            {
                "train_end_year": train_end_year,
                "validation_start_year": train_end_year + 1,
                "test_start_year": test_start_year,
                **{
                    name + "_bankrupt": int(counts.loc[name, 1]) for name in SPLIT_NAMES
                },
                **{
                    name + "_nonbankrupt": int(counts.loc[name, 0])
                    for name in SPLIT_NAMES
                },
                "adequate": acceptable,
            }
        )
        if acceptable:
            selected_audit = audit
            TRAIN_END_YEAR = train_end_year
            VALIDATION_START_YEAR = train_end_year + 1
            TEST_START_YEAR = test_start_year
            break
    if selected_audit is not None:
        break
cutoff_trials = pd.DataFrame(cutoff_trials)
if selected_audit is None:
    raise ValueError(
        "No chronological design meets the configured minimums. Review thresholds/year range before modeling; do not fall back to random splitting."
    )
VALIDATION_END_YEAR = TEST_START_YEAR - 1
selected = selected_audit.loc[selected_audit.split.notna()].copy()

In [4]:
# Document exclusions and how company grouping affects original matching links.
exclusion_summary = (
    selected_audit.loc[selected_audit.split.isna()]
    .groupby(["nominal_split", "exclusion_reason", "target"])
    .size()
    .rename("events")
    .reset_index()
)
split_summary = selected.groupby("split").agg(
    events=("metadata_event_id", "size"),
    companies=("company_id", "nunique"),
    company_groups=("company_group_id", "nunique"),
    bankrupt_events=("target", "sum"),
    first_prediction=("prediction_date", "min"),
    last_prediction=("prediction_date", "max"),
    last_outcome=("metadata_outcome_end_date", "max"),
)
split_summary["nonbankrupt_events"] = (
    split_summary.events - split_summary.bankrupt_events
)
split_summary = split_summary.reindex(SPLIT_NAMES)

# Treat matching JSON as audit metadata; do not restore unavailable cross-fold pairs.
unique_pairs = {}
for value in events.metadata_matching_records_json:
    for record in json.loads(value):
        pair_id = record["matching_pair_id"]
        if pair_id in unique_pairs and unique_pairs[pair_id] != record:
            raise ValueError("Inconsistent matching pair copies")
        unique_pairs[pair_id] = record
pair_rows = []
source_keys = set(pd.MultiIndex.from_frame(events[KEYS]).tolist())
selected_splits = selected.set_index(KEYS)["split"]
for record in unique_pairs.values():
    bank_key = (
        record["bankrupt_company_id"],
        pd.Timestamp(record["bankrupt_prediction_date"]),
    )
    control_key = (
        record["control_company_id"],
        pd.Timestamp(record["control_pseudo_event_date"]),
    )
    bank_split = selected_splits.get(bank_key)
    control_split = selected_splits.get(control_key)
    if bank_split is None or control_split is None:
        status = "excluded_member"
    elif bank_split != control_split:
        status = "cross_split_pair"
    else:
        status = "within_split_pair"
    pair_rows.append(
        {
            "matching_pair_id": record["matching_pair_id"],
            "bankrupt_company_id": bank_key[0],
            "bankrupt_prediction_date": bank_key[1],
            "control_company_id": control_key[0],
            "control_pseudo_event_date": control_key[1],
            "bankrupt_split": bank_split,
            "control_split": control_split,
            "pair_status": status,
            "both_members_in_source_dataset": bank_key in source_keys
            and control_key in source_keys,
        }
    )
matching_split_audit = pd.DataFrame(pair_rows)
within_pairs = matching_split_audit.loc[
    matching_split_audit.pair_status.eq("within_split_pair")
]
positive_keys = pd.MultiIndex.from_frame(selected.loc[selected.target.eq(1), KEYS])
positive_keys.names = ["bankrupt_company_id", "bankrupt_prediction_date"]
within_counts = (
    within_pairs.groupby(["bankrupt_company_id", "bankrupt_prediction_date"])
    .control_company_id.nunique()
    .reindex(positive_keys, fill_value=0)
)
matched_set_coverage = (
    within_counts.rename("controls_in_same_split")
    .reset_index()
    .rename(
        columns={
            "bankrupt_company_id": "company_id",
            "bankrupt_prediction_date": "prediction_date",
        }
    )
)
matched_set_coverage = matched_set_coverage.merge(
    selected[KEYS + ["split"]], on=KEYS, how="left", validate="one_to_one"
)

In [5]:
# Formal split sanity checks, completed before writing any ID file.
outputs = {}
for name in SPLIT_NAMES:
    outputs[name] = (
        selected.loc[selected.split.eq(name), ID_COLUMNS]
        .sort_values(KEYS, kind="stable")
        .reset_index(drop=True)
    )
    frame = outputs[name]
    lo, hi = {
        "train": (START_YEAR, TRAIN_END_YEAR),
        "validation": (VALIDATION_START_YEAR, VALIDATION_END_YEAR),
        "test": (TEST_START_YEAR, END_YEAR),
    }[name]

# Also verify security strings explicitly; missing securities use company-ID groups.
canonical_securities = (
    events.metadata_bloomberg_identifier.astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)
security_sets = {
    name: set(
        canonical_securities.loc[selected_audit.split.eq(name)]
        .dropna()
        .loc[lambda s: s.ne("")]
    )
    for name in SPLIT_NAMES
}
validation_boundary = pd.Timestamp(year=VALIDATION_START_YEAR, month=1, day=1)
test_boundary = pd.Timestamp(year=TEST_START_YEAR, month=1, day=1)
# Group ownership is derived only from already eligible rows, before the grouping exclusion.
eligible_reasons = (
    selected_audit.exclusion_reason.isna()
    | selected_audit.exclusion_reason.eq("company_reserved_for_later_split").fillna(
        False
    )
)
latest_groups = (
    selected_audit.loc[eligible_reasons]
    .groupby("company_group_id")
    .nominal_split_rank.max()
)
all_exported = pd.concat(outputs.values(), ignore_index=True)
source_rows = (
    events.set_index(KEYS)
    .reindex(pd.MultiIndex.from_frame(all_exported[KEYS]))
    .reset_index(drop=True)
)

# Export only membership/label/date metadata; features remain in the unprocessed CSV.
for name, frame in outputs.items():
    output_path = PROCESSED / f"{name}_ids.csv"
    frame.to_csv(output_path, index=False, date_format="%Y-%m-%d")

In [6]:
# Build expanding-window folds before the untouched final test period.
# This cell exports membership only. For every fold, fit preprocessing and the
# model on that fold's train rows, then transform/evaluate its validation rows.
# Never reuse the fixed train/validation matrices for walk-forward evaluation.
WF_INITIAL_TRAIN_YEARS = 10
WF_MIN_VALIDATION_BANKRUPTCIES = 10
WF_MIN_VALIDATION_CONTROLS = 10
WF_MIN_TRAIN_BANKRUPTCIES = 20
WF_MIN_TRAIN_CONTROLS = 20

WF_INITIAL_TRAIN_END_YEAR = START_YEAR + WF_INITIAL_TRAIN_YEARS - 1
WF_DEVELOPMENT_END_YEAR = VALIDATION_END_YEAR
fold_rows = []
fold_summaries = []
fold_number = 1
train_end_year = WF_INITIAL_TRAIN_END_YEAR

while train_end_year < WF_DEVELOPMENT_END_YEAR:
    validation_start_year = train_end_year + 1
    chosen_validation_end = None
    validation_candidates = None
    for candidate_end_year in range(validation_start_year, WF_DEVELOPMENT_END_YEAR + 1):
        start_date = pd.Timestamp(validation_start_year, 1, 1)
        end_date = pd.Timestamp(candidate_end_year + 1, 1, 1)
        candidate = events.loc[
            events.prediction_date.ge(start_date)
            & events.prediction_date.lt(end_date)
            & events.outcome_complete
        ].copy()
        counts = candidate.target.value_counts()
        if (
            int(counts.get(1, 0)) >= WF_MIN_VALIDATION_BANKRUPTCIES
            and int(counts.get(0, 0)) >= WF_MIN_VALIDATION_CONTROLS
        ):
            remaining = events.loc[
                events.prediction_year.gt(candidate_end_year)
                & events.prediction_year.le(WF_DEVELOPMENT_END_YEAR)
                & events.outcome_complete
            ]
            remaining_counts = remaining.target.value_counts()
            if candidate_end_year < WF_DEVELOPMENT_END_YEAR and (
                int(remaining_counts.get(1, 0)) < WF_MIN_VALIDATION_BANKRUPTCIES
                or int(remaining_counts.get(0, 0)) < WF_MIN_VALIDATION_CONTROLS
            ):
                candidate_end_year = WF_DEVELOPMENT_END_YEAR
                candidate = events.loc[
                    events.prediction_date.ge(start_date)
                    & events.prediction_date.lt(
                        pd.Timestamp(candidate_end_year + 1, 1, 1)
                    )
                    & events.outcome_complete
                ].copy()
            chosen_validation_end = candidate_end_year
            validation_candidates = candidate
            break
    if validation_candidates is None:
        break

    validation_start = pd.Timestamp(validation_start_year, 1, 1)
    training = events.loc[
        events.prediction_date.ge(pd.Timestamp(START_YEAR, 1, 1))
        & events.prediction_date.lt(validation_start)
        & events.metadata_outcome_end_date.lt(validation_start)
        & events.outcome_complete
    ].copy()
    validation_groups = set(validation_candidates.company_group_id)
    training = training.loc[~training.company_group_id.isin(validation_groups)].copy()
    train_counts = training.target.value_counts()
    if (
        int(train_counts.get(1, 0)) < WF_MIN_TRAIN_BANKRUPTCIES
        or int(train_counts.get(0, 0)) < WF_MIN_TRAIN_CONTROLS
    ):
        raise ValueError(
            f"Walk-forward fold {fold_number} has too little training data after purge/grouping"
        )
    if set(training.company_group_id).intersection(validation_groups):
        raise ValueError(
            f"Walk-forward fold {fold_number} leaks company groups across train and validation"
        )

    fold_name = f"fold_{fold_number}"
    for split_name, frame in [
        ("train", training),
        ("validation", validation_candidates),
    ]:
        exported = frame[
            [
                "company_id",
                "prediction_date",
                "target",
                "prediction_year",
                "metadata_event_id",
                "company_group_id",
                "metadata_outcome_end_date",
            ]
        ].copy()
        exported.insert(0, "split", split_name)
        exported.insert(0, "fold", fold_name)
        fold_rows.append(exported)
    fold_summaries.append(
        {
            "fold": fold_name,
            "train_start_year": START_YEAR,
            "train_end_year": train_end_year,
            "validation_start_year": validation_start_year,
            "validation_end_year": chosen_validation_end,
            "train_rows": len(training),
            "train_bankruptcies": int(train_counts.get(1, 0)),
            "train_controls": int(train_counts.get(0, 0)),
            "train_company_groups": training.company_group_id.nunique(),
            "validation_rows": len(validation_candidates),
            "validation_bankruptcies": int(validation_candidates.target.eq(1).sum()),
            "validation_controls": int(validation_candidates.target.eq(0).sum()),
            "validation_company_groups": validation_candidates.company_group_id.nunique(),
            "overlapping_company_groups": 0,
            "train_last_outcome_date": training.metadata_outcome_end_date.max(),
        }
    )
    fold_number += 1
    train_end_year = chosen_validation_end

if not fold_rows:
    raise ValueError("No walk-forward fold met the configured event-count minimums")

walk_forward_ids = pd.concat(fold_rows, ignore_index=True)
final_test_ids = selected.loc[
    selected.split.eq("test"),
    [
        "company_id",
        "prediction_date",
        "target",
        "prediction_year",
        "metadata_event_id",
        "company_group_id",
        "metadata_outcome_end_date",
    ],
].copy()
final_test_ids.insert(0, "split", "test")
final_test_ids.insert(0, "fold", "final_test")
walk_forward_ids = pd.concat([walk_forward_ids, final_test_ids], ignore_index=True)
walk_forward_summary = pd.DataFrame(fold_summaries)
walk_forward_summary = pd.concat(
    [
        walk_forward_summary,
        pd.DataFrame(
            [
                {
                    "fold": "final_test",
                    "train_start_year": START_YEAR,
                    "train_end_year": TRAIN_END_YEAR,
                    "validation_start_year": np.nan,
                    "validation_end_year": np.nan,
                    "train_rows": int((selected.split == "train").sum()),
                    "train_bankruptcies": int(
                        selected.loc[selected.split.eq("train"), "target"].sum()
                    ),
                    "train_controls": int(
                        selected.loc[selected.split.eq("train"), "target"].eq(0).sum()
                    ),
                    "train_company_groups": selected.loc[
                        selected.split.eq("train"), "company_group_id"
                    ].nunique(),
                    "validation_rows": np.nan,
                    "validation_bankruptcies": np.nan,
                    "validation_controls": np.nan,
                    "validation_company_groups": np.nan,
                    "overlapping_company_groups": np.nan,
                    "train_last_outcome_date": selected.loc[
                        selected.split.eq("train"), "metadata_outcome_end_date"
                    ].max(),
                    "test_start_year": TEST_START_YEAR,
                    "test_end_year": END_YEAR,
                    "test_rows": len(final_test_ids),
                    "test_bankruptcies": int(final_test_ids.target.sum()),
                    "test_controls": int(final_test_ids.target.eq(0).sum()),
                }
            ]
        ),
    ],
    ignore_index=True,
    sort=False,
)
walk_forward_summary.to_csv(
    PROCESSED / "walk_forward_fold_summary.csv", index=False, date_format="%Y-%m-%d"
)
walk_forward_ids.to_csv(
    PROCESSED / "walk_forward_fold_ids.csv", index=False, date_format="%Y-%m-%d"
)